In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from scipy import stats
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input, Conv2D, MaxPooling2D, Flatten
from tensorflow.keras.optimizers import SGD
from tensorflow.keras import layers, models

In [ ]:
# -------------------- Load and Prepare Data --------------------
print("Loading cifar10 data...")
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Prepare training data (keep original shape for data splitting)
x_train_original = x_train
y_train_original = y_train

# Prepare test data (for final evaluation)
x_test_cnn = x_test.reshape(-1, 32, 32, 3).astype('float32') / 255.0
y_test_categorical = tf.keras.utils.to_categorical(y_test, 10)

print(f"Original training set shape: {x_train_original.shape}")
print(f"Test set shape: {x_test_cnn.shape}")

In [ ]:
# -------------------- Global Parameter Configuration --------------------
# Total number of clients participating in training in the federated learning system (20 devices/users in total)
NUM_CLIENTS = 20
# Number of classes in the dataset (MNIST has 10 digit classes from 0 to 9)
NUM_CLASSES = 10
# Input feature dimension (For CNN, we use image shape instead of flattening)
IMAGE_SHAPE = (28, 28, 1)
# Size of the time window for calculating the historical performance trend of clients 
# (Accuracy of the last 3 rounds is used to fit the trend via linear regression)
WINDOW_SIZE = 3
# Weight of "historical performance trend" in client selection score 
# (Larger values mean more emphasis on the improvement of clients' past performance)
WEIGHT_HISTORY = 0.6
# Weight of "data quality (similarity to global model)" in client selection score 
# (Reflects the potential contribution of local data to the global model)
WEIGHT_DATA = 0.3
# Weight of "system status (simulated indicators such as device power, network condition, etc.)" 
# in client selection score
WEIGHT_SYS = 0.1
# Number of clients selected from all clients to participate in training per federated learning round (17 out of 20)
NUM_SELECT_CLIENTS = 17
# Coefficient for smoothing current accuracy and historical average accuracy during adaptive aggregation 
# (0.9 means more trust in current performance)
# Formula: smoothed_acc = LAMBDA_SMOOTH * current_acc + (1 - LAMBDA_SMOOTH) * avg_acc
LAMBDA_SMOOTH = 0.9
# Number of local training epochs executed by each selected client (local fine-tuning intensity)
LOCAL_EPOCHS = 1  # Reduce epochs to speed up training
# Learning rate
LEARNING_RATE = 0.01
# Batch size for local training (Affects memory usage and convergence stability)
BATCH_SIZE = 30
# Total communication rounds executed by the global coordinator (server) 
# (i.e., number of federated learning iterations)
NUM_GLOBAL_EPOCHS = 30  
# Set random seeds for TensorFlow and NumPy to ensure experimental reproducibility 
# (Consistent results for the same code run each time)
tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
# -------------------- Utility Functions --------------------
def normalize_scores(scores):
    """Normalize scores to the range [0, 1]"""
    if len(scores) == 0:
        return np.array([])
    min_score = np.min(scores)
    max_score = np.max(scores)
    if max_score - min_score < 1e-6:
        return np.ones_like(scores) / len(scores)
    return (scores - min_score) / (max_score - min_score)

def build_model():
    """Build a CNN model"""
    model = tf.keras.models.Sequential([
        layers.InputLayer(input_shape=(32, 32, 3)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')    
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01), loss='sparse_categorical_crossentropy', metrics=['accuracy'])    
    return model

In [ ]:
def load_and_partition_mnist_non_iid(
    num_clients=10,
    client_ratios=None,  # List of data proportion for each client, e.g., [0.1, 0.05, ..., 0.1]
    alpha=0.5,           # Dirichlet distribution parameter, controls the non-IID degree of class distribution within each client
    val_ratio=0.3        # Validation set ratio
):
    
    # Load data
    x_data = x_train_original
    y_data = y_train_original

    total_samples = len(x_data)

    # Set default ratios (uniform distribution)
    if client_ratios is None:
        client_ratios = [1.0 / num_clients] * num_clients
    else:
        assert len(client_ratios) == num_clients, "Length of client_ratios must equal num_clients"
        assert abs(sum(client_ratios) - 1.0) < 1e-6, "Sum of client_ratios must be 1.0"

    # Calculate the number of samples to allocate to each client (adjust total after rounding)
    client_sizes = [int(ratio * total_samples) for ratio in client_ratios]
    diff = total_samples - sum(client_sizes)
    # Allocate the remainder to the first 'diff' clients (simple handling)
    for i in range(diff):
        client_sizes[i] += 1

    # Build index pool by class
    class_indices = [np.where(y_data == i)[0] for i in range(NUM_CLASSES)]
    used_mask = np.zeros(total_samples, dtype=bool)  # Mark whether samples are allocated

    client_data = []

    for i in range(num_clients):
        needed = client_sizes[i]

        # Generate a class distribution for the current client (Dirichlet distribution)
        class_probs = np.random.dirichlet([alpha] * NUM_CLASSES)

        # Sample according to the class distribution
        selected_indices = []
        remaining = needed

        # Iterate through classes in random order to avoid always sampling from front classes
        class_order = np.random.permutation(NUM_CLASSES)
        for class_id in class_order:
            # Calculate the number of samples to draw from this class
            n_from_class = int(class_probs[class_id] * needed)
            # Ensure at least one sample per class if there are remaining samples to allocate
            if n_from_class == 0 and remaining > 0:
                n_from_class = 1

            # Adjust if the required samples exceed remaining needs
            if n_from_class > remaining:
                n_from_class = remaining

            # Get unused sample indices from this class
            available_indices = class_indices[class_id][~used_mask[class_indices[class_id]]]
            # Take all available samples if insufficient
            n_available = len(available_indices)
            if n_available <= n_from_class:
                selected = available_indices
            else:
                # Randomly select n_from_class samples
                selected = np.random.choice(available_indices, size=n_from_class, replace=False)

            selected_indices.extend(selected)
            used_mask[selected] = True
            remaining -= len(selected)

            if remaining <= 0:
                break

        # If not enough samples are collected, randomly supplement from all remaining samples
        if remaining > 0:
            all_remaining = np.where(~used_mask)[0]
            if len(all_remaining) < remaining:
                # Take all remaining samples if insufficient
                selected = all_remaining
            else:
                selected = np.random.choice(all_remaining, size=remaining, replace=False)
            selected_indices.extend(selected)
            used_mask[selected] = True

        # Ensure we have collected the required number of samples
        assert len(selected_indices) == needed, f"Client {i} sample count error: expected {needed}, actual {len(selected_indices)}"

        # Convert to array
        selected_indices = np.array(selected_indices)
        np.random.shuffle(selected_indices)

        X_local = x_data[selected_indices]
        y_local = y_data[selected_indices]

        # Preprocessing: normalize and reshape to CNN input shape
        X_local = X_local.reshape(-1, 32, 32, 3).astype('float32') / 255.0

        # Split into training/validation sets
        split = int((1 - val_ratio) * len(X_local))
        client_data.append({
            'X_train': X_local[:split],
            'y_train': y_local[:split],
            'X_val': X_local[split:],
            'y_val': y_local[split:],
            'class_probs': class_probs  # Save class distribution (optional)
        })

    # Print statistics
    actual_sizes = [len(d['X_train']) + len(d['X_val']) for d in client_data]
    print(f"[Non-IID Proportional Partition Complete] Total samples: {total_samples}")
    print(f"Client data sizes → min: {min(actual_sizes)}, max: {max(actual_sizes)}, mean: {np.mean(actual_sizes):.1f}")
    print(f"Client data proportions (actual): {[size/total_samples for size in actual_sizes]}")

    return client_data

In [ ]:
# -------------------- Client Class --------------------
class Client:
    def __init__(self, client_id, data_dict):
        self.client_id = client_id
        self.X_train = data_dict['X_train']
        self.y_train = data_dict['y_train']
        self.X_val = data_dict['X_val']
        self.y_val = data_dict['y_val']
        self.model = build_model()
        self.acc_history = []
        self.acc_trend = 0.0
        self.data_similarity = 0.0
        self.sys_score = np.random.uniform(0.7, 1.0)  # Simulated system score
    
    def local_train(self, global_weights):
        """Perform local training using global weights"""
        self.model.set_weights(global_weights)
        self.model.fit(
            self.X_train, 
            self.y_train,
            epochs=LOCAL_EPOCHS, 
            batch_size=BATCH_SIZE, 
            verbose=0,
            validation_data=(self.X_val, self.y_val)
        )
        return self.model.get_weights()
    
    def local_evaluate(self):
        """Evaluate the model on the local validation set"""
        y_pred = np.argmax(self.model.predict(self.X_val, verbose=0), axis=1)
        acc = accuracy_score(self.y_val, y_pred)
        self.acc_history.append(acc)
        
        # Calculate accuracy trend (last WINDOW_SIZE rounds)
        hist = self.acc_history
        if len(hist) >= WINDOW_SIZE:
            x = np.arange(WINDOW_SIZE)
            slope, _, _, _, _ = stats.linregress(x, hist[-WINDOW_SIZE:])
            self.acc_trend = slope
        elif len(hist) >= 2:
            x = np.arange(len(hist))
            slope, _, _, _, _ = stats.linregress(x, hist)
            self.acc_trend = slope
        else:
            self.acc_trend = 0.0
        
        return acc
    
    def get_historical_performance_factor(self):
        """Get historical performance factor"""
        return max(0, self.acc_trend) + 0.1  # Ensure non-negative
    
    def get_data_quality_factor(self):
        """Get data quality factor"""
        return max(0.1, min(1.0, self.data_similarity))  # Limit to [0.1, 1.0]
    
    def get_system_factor(self):
        """Get system factor"""
        return self.sys_score

In [ ]:
# -------------------- Server: FedDAAW --------------------
class ServerDA:
    def __init__(self):
        self.global_model = build_model()
        self.global_weights = self.global_model.get_weights()
        self.global_acc_history = []
    
    def calculate_selection_score(self, clients):
        """Calculate client selection scores"""
        scores = []
        for c in clients:
            hist_factor = c.get_historical_performance_factor()
            data_factor = c.get_data_quality_factor()
            sys_factor = c.get_system_factor()
            
            score = (WEIGHT_HISTORY * hist_factor + 
                    WEIGHT_DATA * data_factor + 
                    WEIGHT_SYS * sys_factor)
            scores.append(score)
        
        return np.array(scores)
    
    def select_clients(self, clients):
        """Select clients based on scores"""
        scores = self.calculate_selection_score(clients)
        probs = normalize_scores(scores)
        probs = probs / probs.sum()  # Normalize to probability distribution
        
        # If all probabilities are 0, select uniformly
        if np.any(np.isnan(probs)) or np.all(probs == 0):
            probs = np.ones(len(clients)) / len(clients)
        
        indices = np.random.choice(
            len(clients), 
            size=min(NUM_SELECT_CLIENTS, len(clients)), 
            replace=False, 
            p=probs
        )
        
        selected = [clients[i] for i in indices]
        print(f"DA selected clients: {[c.client_id for c in selected]}")
        return selected
    
    def calculate_adaptive_weights(self, selected_clients):
        """Calculate adaptive aggregation weights"""
        weights = []
        for c in selected_clients:
            if len(c.acc_history) == 0:
                weights.append(0.1)  # Default weight
            else:
                current = c.acc_history[-1]
                avg = np.mean(c.acc_history) if len(c.acc_history) > 0 else current
                smoothed = LAMBDA_SMOOTH * current + (1 - LAMBDA_SMOOTH) * avg
                weights.append(smoothed)
        
        weights = np.array(weights)
        if np.sum(weights) > 0:
            weights = weights / np.sum(weights)
        else:
            weights = np.ones(len(weights)) / len(weights)
        
        return weights
    
    def update_data_similarity(self, selected_clients, client_weights_list):
        """Update data similarity (based on cosine similarity of first layer weights)"""
        if len(selected_clients) == 0:
            return
        
        # Get global model's first layer weights
        global_first_layer = self.global_weights[0].flatten()
        global_norm = np.linalg.norm(global_first_layer)
        
        for i, (c, cw) in enumerate(zip(selected_clients, client_weights_list)):
            # Get client model's first layer weights
            client_first_layer = cw[0].flatten()
            client_norm = np.linalg.norm(client_first_layer)
            
            if global_norm > 1e-8 and client_norm > 1e-8:
                # Calculate cosine similarity
                similarity = np.dot(global_first_layer, client_first_layer) / (global_norm * client_norm)
                c.data_similarity = max(0, min(1, similarity))  # Limit to [0, 1]
            else:
                c.data_similarity = 0.5  # Default value
    
    def aggregate(self, selected_clients, client_weights_list):
        """Aggregate client models"""
        if len(selected_clients) == 0:
            return
        
        adaptive_w = self.calculate_adaptive_weights(selected_clients)
        print(f"DA aggregation weights: {adaptive_w}")
        
        new_weights = []
        for layer_idx in range(len(self.global_weights)):
            # Weighted aggregation
            weighted_sum = np.zeros_like(self.global_weights[layer_idx])
            for alpha, cw in zip(adaptive_w, client_weights_list):
                weighted_sum += alpha * cw[layer_idx]
            new_weights.append(weighted_sum)
        
        self.global_weights = new_weights
        self.update_data_similarity(selected_clients, client_weights_list)
    
    def evaluate(self, test_data=None, test_labels=None):
        """Evaluate global model"""
        if test_data is None:
            # Use test set
            test_data = x_test_cnn
            test_labels = y_test_categorical
        
        self.global_model.set_weights(self.global_weights)
        y_pred = np.argmax(self.global_model.predict(test_data, verbose=0), axis=1)
        
        if test_labels.ndim > 1:  # If one-hot encoded
            y_true = np.argmax(test_labels, axis=1)
        else:
            y_true = test_labels
            
        acc = accuracy_score(y_true, y_pred)
        self.global_acc_history.append(acc)
        
        return acc

In [ ]:
# -------------------- Server: FedAvg (Baseline) --------------------
class ServerFedAvg:
    def __init__(self):
        self.global_model = build_model()
        self.global_weights = self.global_model.get_weights()
        self.global_acc_history = []
    
    def select_clients(self, clients):
        """Randomly select clients"""
        indices = np.random.choice(
            len(clients), 
            size=min(NUM_SELECT_CLIENTS, len(clients)), 
            replace=False
        )
        selected = [clients[i] for i in indices]
        print(f"FedAvg selected clients: {[c.client_id for c in selected]}")
        return selected
    
    def aggregate(self, client_weights_list):
        """FedAvg aggregation: simple average"""
        if len(client_weights_list) == 0:
            return
        
        new_weights = []
        for layer_idx in range(len(self.global_weights)):
            avg_layer = np.zeros_like(self.global_weights[layer_idx])
            for cw in client_weights_list:
                avg_layer += cw[layer_idx]
            avg_layer = avg_layer / len(client_weights_list)
            new_weights.append(avg_layer)
        
        self.global_weights = new_weights
    
    def evaluate(self, test_data=None, test_labels=None):
        """Evaluate global model"""
        if test_data is None:
            # Use test set
            test_data = x_test_cnn
            test_labels = y_test_categorical
        
        self.global_model.set_weights(self.global_weights)
        y_pred = np.argmax(self.global_model.predict(test_data, verbose=0), axis=1)
        
        if test_labels.ndim > 1:  # If one-hot encoded
            y_true = np.argmax(test_labels, axis=1)
        else:
            y_true = test_labels
            
        acc = accuracy_score(y_true, y_pred)
        self.global_acc_history.append(acc)
        
        return acc

In [ ]:
## -------------------- Main Training Process --------------------
da_accuracies = []
fedavg_accuracies = []

def main():
    print("\n" + "="*60)
    print("Federated Learning Experiment: Dynamic-Adaptive vs FedAvg")
    print("="*60)
    
    # Non-IID data partitioning
    print("\nPerforming Non-IID data partitioning...")
    fixed_ratios = [0.1, 0.1, 0.1, 0.1, 0.05, 0.05, 0.05, 0.1, 0.1, 0.1, 
                    0.005, 0.005, 0.02, 0.03, 0.002, 0.003, 0.002, 0.003, 0.04, 0.04]
    client_datasets = load_and_partition_mnist_non_iid(
        num_clients=NUM_CLIENTS, 
        alpha=0.5,  # Dirichlet distribution parameter, controls Non-IID degree
        client_ratios=fixed_ratios,
        val_ratio=0.2
    )
    
    # Create two groups of clients (same data, independent models)
    clients_da = [Client(i, client_datasets[i]) for i in range(NUM_CLIENTS)]
    clients_fedavg = [Client(i, client_datasets[i]) for i in range(NUM_CLIENTS)]
    
    # Initialize servers
    server_da = ServerDA()
    server_fedavg = ServerFedAvg()
    
    print(f"\nStarting federated learning training...")
    print(f"Global epochs: {NUM_GLOBAL_EPOCHS}, Clients selected per round: {NUM_SELECT_CLIENTS}")
    print(f"Local training epochs: {LOCAL_EPOCHS}, Batch size: {BATCH_SIZE}")
    
    for epoch in range(NUM_GLOBAL_EPOCHS):
        print(f"\n{'='*50}")
        print(f"Global Epoch {epoch+1}/{NUM_GLOBAL_EPOCHS}")
        print(f"{'='*50}")
        
        # ---------- Dynamic-Adaptive ----------
        print("\n[DA Method]")
        selected_da = server_da.select_clients(clients_da)
        
        # Client local training
        weights_da = []
        for client in selected_da:
            client_weights = client.local_train(server_da.global_weights)
            weights_da.append(client_weights)
        
        # Client local evaluation
        for client in selected_da:
            client.local_evaluate()
        
        # Server aggregation
        server_da.aggregate(selected_da, weights_da)
        
        # Global evaluation
        acc_da = server_da.evaluate()
        da_accuracies.append(acc_da)
        
        # ---------- FedAvg ----------
        print("\n[FedAvg Method]")
        selected_fed = server_fedavg.select_clients(clients_fedavg)
        
        # Client local training
        weights_fed = []
        for client in selected_fed:
            client_weights = client.local_train(server_fedavg.global_weights)
            weights_fed.append(client_weights)
        
        # Client local evaluation
        for client in selected_fed:
            client.local_evaluate()
        
        # Server aggregation
        server_fedavg.aggregate(weights_fed)
        
        # Global evaluation
        acc_fed = server_fedavg.evaluate()
        fedavg_accuracies.append(acc_fed)
        
        print(f"\nEpoch {epoch+1} Results:")
        print(f"  DA Accuracy: {acc_da:.4f}")
        print(f"  FedAvg Accuracy: {acc_fed:.4f}")
        
        # Early stopping check (if accuracy doesn't improve for 3 consecutive epochs)
        if epoch >= 3:
            if (da_accuracies[-1] <= da_accuracies[-2] <= da_accuracies[-3] and
                fedavg_accuracies[-1] <= fedavg_accuracies[-2] <= fedavg_accuracies[-3]):
                print("\nConvergence detected, early stopping training")
                break

if __name__ == "__main__":
    main()

In [ ]:
if 1==1:
# ---------- Result Visualization ----------
    epochs_trained = len(da_accuracies)
    
    plt.figure(figsize=(12, 5))
    
    # Subplot 1: Accuracy curve
    plt.subplot(1, 2, 1)
    plt.plot(range(1, epochs_trained + 1), da_accuracies, 'b-o', linewidth=2, markersize=6, label='FedDAAW')
    plt.plot(range(1, epochs_trained + 1), fedavg_accuracies, 'r--s', linewidth=2, markersize=6, label='FedAvg')
    plt.xlabel('Rounds of Non-IID CIFAR-10 Training', fontsize=12)
    plt.ylabel('Validation Accuracy', fontsize=12)
    plt.title('Accuracy Variation Curve of FedDAAW and FedAvg', fontsize=12)
    plt.legend(fontsize=11)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.ylim([0.1, 0.5])
    
    # Subplot 2: Final accuracy comparison
    plt.subplot(1, 2, 2)
    methods = ['FedDAAW', 'FedAvg']
    final_accs = [da_accuracies[-1], fedavg_accuracies[-1]]
    colors = ['blue', 'red']
    
    bars = plt.bar(methods, final_accs, color=colors, alpha=0.7, edgecolor='black')
    plt.ylabel('Test Accuracy', fontsize=12)
    plt.title('Final Test Accuracy Comparison of FedDAAW and FedAvg', fontsize=12)
    plt.ylim([0.1, 0.5])
    
    # Display accuracy values on the bar chart
    for bar, acc in zip(bars, final_accs):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{acc:.4f}', ha='center', va='bottom', fontsize=11)
    
    plt.grid(True, linestyle='--', alpha=0.7, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    # ---------- Print Final Results ----------
    print("\n" + "="*60)
    print("Final Experimental Results")
    print("="*60)
    print(f"FedDynAda Final Accuracy: {da_accuracies[-1]:.4f}")
    print(f"FedAvg Final Accuracy: {fedavg_accuracies[-1]:.4f}")
    
    improvement = da_accuracies[-1] - fedavg_accuracies[-1]
    if improvement > 0:
        print(f"FedDynAda outperforms FedAvg by: {improvement:.4f} ({improvement*100:.2f}%)")
    elif improvement < 0:
        print(f"FedAvg outperforms FedDynAda by: {abs(improvement):.4f} ({abs(improvement)*100:.2f}%)")
    else:
        print("Both methods achieve identical performance")
    
    # Calculate convergence speed (number of rounds to reach 90% accuracy)
    da_90_idx = next((i for i, acc in enumerate(da_accuracies) if acc >= 0.9), None)
    fedavg_90_idx = next((i for i, acc in enumerate(fedavg_accuracies) if acc >= 0.9), None)
    
    if da_90_idx is not None:
        print(f"FedDynAda reaches 90% accuracy at round {da_90_idx + 1}")
    else:
        print("FedDynAda does not reach 90% accuracy during training")
        
    if fedavg_90_idx is not None:
        print(f"FedAvg reaches 90% accuracy at round {da_90_idx + 1}")  
    else:
        print("FedAvg does not reach 90% accuracy during training")